In [ ]:
get_ipython().system('pip install openai-whisper')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=ada3b153085ba924daee19c96b3ea0d504e1cdf9fa487219399b092dce9b2962
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
!pip install pyctcdecode kenlm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 kB 11.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.3/540.3 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.8 MB/s eta 0:00:00
  Created wheel for kenlm: filename=kenlm-0.3.0-cp312-cp312-linux_x86_64.whl size=3188056 sha256=4e945d5d2bce149b377aff7c49a004ae8ec5da85de097fee26ee503476322a13
  Stored in directory: /root/.cache/pip/wheels/0c/e6/ad/18d2d3f1290a6be6a14a24e90f2b78bb3300aab3852ceb06a6
Successfully built kenlm
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [ ]:
import time
from pathlib import Path

import torch
import os,csv,json

In [ ]:
import whisper

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nLoading Whisper Large-v3 on {DEVICE} ...")

model = whisper.load_model("large-v3", download_root="./models").to(DEVICE)

print("✓ Model loaded")


Loading Whisper Large-v3 on cuda ...


100%|██████████████████████████████████████| 2.88G/2.88G [00:16<00:00, 192MiB/s]


✓ Model loaded


In [ ]:
AUDIO_DIR   = "/content/files"
RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

AUDIO_FILES = {
    "Koramangala"    : "Koramangala .m4a",
    "Indiranagar"    : "Indiranagar .m4a",
    "Whitefield"     : "Whitefield .m4a",
    "Electronic City": "Electronic city .m4a",
    "Marathahalli"   : "Marathahalli .m4a",
    "Jayanagar"      : "Jayanagar .m4a",
    "Rajajinagar"    : "rajajinagar .m4a",
    "Hebbal"         : "Hebbel .m4a",
    "Yelahanka"      : "Yelahanka .m4a",
    "Banashankari"   : "Banashankari .m4a",
    "HSR Layout"     : "HSR layout .m4a",
    "BTM Layout"     : "BTM layout .m4a",
    "Majestic"       : "majestic .m4a",
    "Silk Board"     : "silk board.m4a",
    "Bellandur"      : "Bellandur .m4a",
    "Sarjapur"       : "Sarjapur .m4a",
    "Bommanahalli"   : "Bommanahalli .m4a",
    "KR Puram"       : "KR Puram.m4a",
    "Peenya"         : "peenya.m4a",
    "Yeshwanthpur"   : "Yeshwantpur .m4a",
    "Indiranagar_2"  : "Indiranagar_2.m4a",
    "Bellandur_2"    : "Bellandur2.m4a",
    "Sarjapur_2"     : "Sarjapur_2.m4a",
    "Yeshwanthpur_2" : "yeshwantpur 2.m4a",
}

GROUND_TRUTH = {
    "Koramangala"    : "haan main koramangala mein rehta hoon",
    "Indiranagar"    : "mera ghar indiranagar ke paas hai",
    "Whitefield"     : "haan main whitefield se aata hoon",
    "Electronic City": "mera office electronic city mein hi hai",
    "Marathahalli"   : "marathahalli bridge ke paas drop karo",
    "Jayanagar"      : "jayanagar mein ek room dhundh raha hoon",
    "Rajajinagar"    : "rajajinagar se bus pakadta hoon main",
    "Hebbal"         : "hebbal flyover ke paas rehta hoon",
    "Yelahanka"      : "yelahanka new town mein shift hua hoon",
    "Banashankari"   : "banashankari temple ke paas ghar hai mera",
    "HSR Layout"     : "hsr layout sector 2 mein hoon main abhi",
    "BTM Layout"     : "btm layout first stage jaante ho",
    "Majestic"       : "majestic se bmtc leke jaata hoon",
    "Silk Board"     : "silk board par traffic bahut hai",
    "Bellandur"      : "bellandur road pe flat liya hai",
    "Sarjapur"       : "sarjapur road par kaam karta hoon",
    "Bommanahalli"   : "bommanahalli se koramangala 20 min hai",
    "KR Puram"       : "kr puram railway station ke paas khada hoon jaldi aaja",
    "Peenya"         : "peenya industrial area mein factory hai",
    "Yeshwanthpur"   : "yeshwanthpur se train pakadni hai mereko jaldi aa",
    "Indiranagar_2"  : "mera ghar indiranagar ke paas hi hai",
    "Bellandur_2"    : "bellandur lake ke paas flat liya hai",
    "Sarjapur_2"     : "sarjapur road pe khada hun jaldi aaja",
    "Yeshwanthpur_2" : "yeshwantpur se train leni hai jaldi aa",
}

AUDIO_CONDITIONS = {
    "Koramangala": "quiet", "Indiranagar": "quiet", "Whitefield": "quiet",
    "Electronic City": "quiet", "Marathahalli": "street_noise",
    "Jayanagar": "street_noise", "Rajajinagar": "street_noise",
    "Hebbal": "street_noise", "Yelahanka": "background_noise",
    "Banashankari": "background_noise", "HSR Layout": "rushed",
    "BTM Layout": "rushed", "Majestic": "whispered", "Silk Board": "whispered",
    "Bellandur": "phone_quality", "Sarjapur": "phone_quality",
    "Bommanahalli": "noisy_fast", "KR Puram": "noisy_fast",
    "Peenya": "quiet_mumbled", "Yeshwanthpur": "quiet",
    "Indiranagar_2": "street_noise", "Bellandur_2": "street_noise",
    "Sarjapur_2": "street_noise", "Yeshwanthpur_2": "street_noise",
}

SPEAKER = {
    "Koramangala": "self", "Indiranagar": "self", "Whitefield": "self",
    "Electronic City": "self", "Marathahalli": "self", "Jayanagar": "self",
    "Rajajinagar": "self", "Hebbal": "self", "Yelahanka": "self",
    "Banashankari": "self", "HSR Layout": "self", "BTM Layout": "self",
    "Majestic": "self", "Silk Board": "self", "Bellandur": "self",
    "Sarjapur": "self", "Bommanahalli": "self", "KR Puram": "self",
    "Peenya": "self", "Yeshwanthpur": "self",
    "Indiranagar_2": "friend", "Bellandur_2": "friend",
    "Sarjapur_2": "self", "Yeshwanthpur_2": "self_modulated",
}


In [ ]:
def wer(ref, hyp):
    r = ref.split()
    h = hyp.split()
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            if r[i-1] == h[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = min(d[i-1][j], d[i][j-1], d[i-1][j-1]) + 1
    return d[len(r)][len(h)] / max(len(r), 1)


In [ ]:
def cer(ref, hyp):
    r = list(ref)
    h = list(hyp)
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            if r[i-1] == h[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = min(d[i-1][j], d[i][j-1], d[i-1][j-1]) + 1
    return d[len(r)][len(h)] / max(len(r), 1)

In [ ]:
def entity_match(canonical_locality, hypothesis):

    can = canonical_locality.lower().replace(" ", "")
    hyp_clean = hypothesis.lower().replace(" ", "")
    hyp_words = hypothesis.lower().split()

    exact = can in hyp_clean

    fuzzy = any(can in w or w in can for w in hyp_words)

    def lcs_len(s1, s2):
        dp = [[0]*(len(s2)+1) for _ in range(len(s1)+1)]
        for i in range(len(s1)):
            for j in range(len(s2)):
                if s1[i] == s2[j]:
                    dp[i+1][j+1] = dp[i][j] + 1
                else:
                    dp[i+1][j+1] = max(dp[i+1][j], dp[i][j+1])
        return dp[-1][-1]
    ratio = lcs_len(can, hyp_clean) / max(len(can), 1)
    return {"exact": exact, "fuzzy": fuzzy, "ratio": round(ratio, 3)}

In [ ]:
def aggregate_metrics(rows):

    wers = [r["wer"] for r in rows]
    cers = [r["cer"] for r in rows]
    exacts = [r["entity_exact"] for r in rows]
    fuzzys = [r["entity_fuzzy"] for r in rows]
    return {
        "avg_wer": round(sum(wers)/len(wers), 3),
        "avg_cer": round(sum(cers)/len(cers), 3),
        "entity_exact_pct": round(100 * sum(exacts)/len(exacts), 1),
        "entity_fuzzy_pct": round(100 * sum(fuzzys)/len(fuzzys), 1),
        "num_samples": len(rows),
    }

In [ ]:
def evaluate(model_name, transcriptions):
    rows = []
    for locality, result in transcriptions.items():
        canonical = locality.replace("_2", "").replace("_", " ").strip()
        gt    = GROUND_TRUTH.get(locality, "")
        hyp   = result.get("transcript", "")
        error = result.get("error")
        if error:
            rows.append({
                "model": model_name, "locality": locality, "canonical": canonical,
                "condition": AUDIO_CONDITIONS.get(locality, "unknown"),
                "speaker": SPEAKER.get(locality, "unknown"),
                "ground_truth": gt, "hypothesis": "",
                "wer": 1.0, "cer": 1.0,
                "entity_exact": False, "entity_fuzzy": False, "entity_ratio": 0.0,
                "latency_s": 0.0, "error": error,
            })
            continue
        em = entity_match(canonical, hyp)
        rows.append({
            "model": model_name, "locality": locality, "canonical": canonical,
            "condition": AUDIO_CONDITIONS.get(locality, "unknown"),
            "speaker": SPEAKER.get(locality, "unknown"),
            "ground_truth": gt, "hypothesis": hyp,
            "wer": round(wer(gt, hyp), 3),
            "cer": round(cer(gt, hyp), 3),
            "entity_exact": em["exact"],
            "entity_fuzzy": em["fuzzy"],
            "entity_ratio": em["ratio"],
            "latency_s": result.get("latency_s", 0.0),
            "error": None,
        })
    return rows


In [ ]:
def print_summary_table(summary):
    print("\n" + "=" * 70)
    print(f"{'Model':<25} {'Avg WER':>8} {'Avg CER':>8} {'EMR Exact':>10} {'EMR Fuzzy':>10}")
    print("-" * 70)
    for model, s in summary.items():
        print(f"{model:<25} {s['avg_wer']:>8.3f} {s['avg_cer']:>8.3f} "
              f"{s['entity_exact_pct']:>9.1f}% {s['entity_fuzzy_pct']:>9.1f}%")
    print("=" * 70)

def print_condition_breakdown(all_rows):
    from collections import defaultdict
    data = defaultdict(lambda: defaultdict(list))
    for r in all_rows:
        data[r["model"]][r["condition"]].append(r["wer"])
    conditions = sorted({r["condition"] for r in all_rows})
    models = sorted(data.keys())
    print("\n── WER by Audio Condition ──")
    header = f"{'Condition':<20}" + "".join(f"{m[:15]:>16}" for m in models)
    print(header)
    print("-" * len(header))
    for cond in conditions:
        row_str = f"{cond:<20}"
        for model in models:
            vals = data[model][cond]
            avg = sum(vals) / len(vals) if vals else float("nan")
            row_str += f"{avg:>16.3f}"
        print(row_str)


In [ ]:
def print_speaker_breakdown(all_rows):
    from collections import defaultdict
    data = defaultdict(lambda: defaultdict(list))
    for r in all_rows:
        data[r["model"]][r["speaker"]].append(r["wer"])
    speakers = sorted({r["speaker"] for r in all_rows})
    models = sorted(data.keys())
    print("\n── WER by Speaker ──")
    header = f"{'Speaker':<20}" + "".join(f"{m[:15]:>16}" for m in models)
    print(header)
    print("-" * len(header))
    for spk in speakers:
        row_str = f"{spk:<20}"
        for model in models:
            vals = data[model][spk]
            avg = sum(vals) / len(vals) if vals else float("nan")
            row_str += f"{avg:>16.3f}"
        print(row_str)


In [ ]:
def transcribe_all(audio_dir, audio_files, language="hi"):
    results = {}
    for loc, fname in audio_files.items():
        path = os.path.join(audio_dir, fname)
        if not os.path.exists(path):
            print(f"⚠ File missing: {path}, skipping.")
            results[loc] = {"transcript": "", "latency_s": 0.0, "error": "file not found"}
            continue
        start = time.time()
        try:
            out = model.transcribe(path, language=language, verbose=False)
            text = out["text"].strip()
            latency = time.time() - start
            results[loc] = {"transcript": text, "latency_s": round(latency, 3), "error": None}
        except Exception as e:
            latency = time.time() - start
            results[loc] = {"transcript": "", "latency_s": round(latency, 3), "error": str(e)}
    return results

print("Transcribing all audio samples with Whisper Large‑v3 ...")
transcriptions = transcribe_all(AUDIO_DIR, AUDIO_FILES)


Transcribing all audio samples with Whisper Large‑v3 ...


100%|██████████| 420/420 [00:02<00:00, 199.51frames/s]


In [ ]:
rows = evaluate("whisper-large-v3", transcriptions)
summary = {"whisper-large-v3": aggregate_metrics(rows)}

print_summary_table(summary)
print_condition_breakdown(rows)
print_speaker_breakdown(rows)

print("\n── Top 5 Failures (highest WER) ──")
worst = sorted(rows, key=lambda x: x["wer"], reverse=True)[:5]
for r in worst:
    print(f"  {r['locality']:<22} WER={r['wer']:.2f}  "
          f"GT: '{r['ground_truth']}'  →  HYP: '{r['hypothesis']}'")



Model                      Avg WER  Avg CER  EMR Exact  EMR Fuzzy
----------------------------------------------------------------------
whisper-large-v3             1.083    0.860       4.2%       4.2%

── WER by Audio Condition ──
Condition            whisper-large-v
------------------------------------
background_noise               1.071
noisy_fast                     1.134
phone_quality                  1.000
quiet                          1.031
quiet_mumbled                  1.000
rushed                         1.021
street_noise                   1.131
whispered                      1.167

── WER by Speaker ──
Speaker              whisper-large-v
------------------------------------
friend                         1.071
self                           1.074
self_modulated                 1.286

── Top 5 Failures (highest WER) ──
  Majestic               WER=1.33  GT: 'majestic se bmtc leke jaata hoon'  →  HYP: 'मैं जे स्टिक से BMTC लेकर जाता हूँ'
  Sarjapur_2             WER=1.29

In [ ]:
raw_csv = os.path.join(RESULTS_DIR, "whisper_large_v3_raw.csv")
with open(raw_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(f" Raw results saved → {raw_csv}")

# Summary CSV
summary_csv = os.path.join(RESULTS_DIR, "whisper_large_v3_summary.csv")
summary_rows = [{"model": model_name, **stats} for model_name, stats in summary.items()]
with open(summary_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=summary_rows[0].keys())
    writer.writeheader()
    writer.writerows(summary_rows)
print(f" Summary saved → {summary_csv}")

# Raw transcriptions JSON
raw_json = os.path.join(RESULTS_DIR, "whisper_large_v3_transcriptions.json")
with open(raw_json, "w", encoding="utf-8") as f:
    json.dump(
        {loc: t["transcript"] for loc, t in transcriptions.items()},
        f, indent=2, ensure_ascii=False
    )
print(f"✓ Raw transcriptions → {raw_json}")



 Raw results saved → /content/results/whisper_large_v3_raw.csv
 Summary saved → /content/results/whisper_large_v3_summary.csv
✓ Raw transcriptions → /content/results/whisper_large_v3_transcriptions.json


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Login to Hugging Face Hub
login(hf_token)

In [ ]:
from transformers import pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="Harveenchadha/vakyansh-wav2vec2-hindi-him-4200"
)
result = pipe("/content/files/Bellandur2.m4a")
print(result)
text = result["text"].replace("<s>", "").strip()
print(text)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/213 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

{'text': '<s>ब<s>ै<s>ल<s>ड<s>ू<s> <s>ल<s>े<s>क<s>े<s> <s>प<s>ा<s>स<s> <s>फ<s>्<s>ल<s>ै<s>ट<s> <s>ल<s>ी<s>है<s> <s>'}
बैलडू लेके पास फ्लैट लीहै


In [ ]:
import time
import torch
import torchaudio
from transformers import pipeline

# Load model once
pipe = pipeline(
    "automatic-speech-recognition",
    model="Harveenchadha/vakyansh-wav2vec2-hindi-him-4200",
    device=0 if torch.cuda.is_available() else -1
)

AUDIO_DIR = "/content/files"  # update path if different

AUDIO_FILES = {
    "Koramangala": "Koramangala .m4a",
    "Indiranagar": "Indiranagar .m4a",
    "Whitefield": "Whitefield .m4a",
    "Electronic City": "Electronic city .m4a",
    "Marathahalli": "Marathahalli .m4a",
    "Jayanagar": "Jayanagar .m4a",
    "Rajajinagar": "rajajinagar .m4a",
    "Hebbal": "Hebbel .m4a",
    "Yelahanka": "Yelahanka .m4a",
    "Banashankari": "Banashankari .m4a",
    "HSR Layout": "HSR layout .m4a",
    "BTM Layout": "BTM layout .m4a",
    "Majestic": "majestic .m4a",
    "Silk Board": "silk road.m4a",
    "Bellandur": "Bellandur .m4a",
    "Sarjapur": "Sarjapur .m4a",
    "Bommanahalli": "Bommanahalli .m4a",
    "KR Puram": "KR Puram.m4a",
    "Peenya": "peenya.m4a",
    "Yeshwanthpur": "Yeshwantpur .m4a",
    "Indiranagar_2": "Indiranagar_2.m4a",
    "Bellandur_2": "Bellandur2.m4a",
    "Sarjapur_2": "Sarjapur_2.m4a",
    "Yeshwanthpur_2": "yeshwantpur 2.m4a",
}

results = {}
for locality, fname in AUDIO_FILES.items():
    path = f"{AUDIO_DIR}/{fname}"
    print(f"  [Vakyansh] {locality} ...", end=" ", flush=True)
    try:
        t0 = time.time()
        out = pipe(path)
        latency = round(time.time() - t0, 3)
        text = out["text"].replace("<s>", "").strip()
        results[locality] = {"transcript": text, "latency_s": latency, "error": None}
        print(f"✓  → {text[:50]}")
    except Exception as e:
        results[locality] = {"transcript": "", "latency_s": 0, "error": str(e)}
        print(f"✗  {e}")

import json
with open("/content/vakyansh_transcriptions.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("\nSaved → vakyansh_transcriptions.json")

Loading weights:   0%|          | 0/213 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

  [Vakyansh] Koramangala ... ✓  → मैं   मेंरहता हूँ
  [Vakyansh] Indiranagar ... ✓  → मेरा घरइंद्र नगर के पास
  [Vakyansh] Whitefield ... ✓  → हां मैं फाइट फील्ड  आता हूं
  [Vakyansh] Electronic City ... ✓  → मेरा ऑफिस इलेक्ट्रॉनिक सिटी में हीहै
  [Vakyansh] Marathahalli ... ✓  → मरथली  केपास जॉब करो
  [Vakyansh] Jayanagar ... ✓  → जयनगर
  [Vakyansh] Rajajinagar ... ✓  → राजाजी नगर से बसपकड़ता हूं
  [Vakyansh] Hebbal ... ✓  → 
  [Vakyansh] Yelahanka ... ✓  → में शिफ्ट
  [Vakyansh] Banashankari ... ✓  → वशंकर  टेम्पल के पास घरमेरा
  [Vakyansh] HSR Layout ... 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✓  → 
  [Vakyansh] BTM Layout ... ✓  → बीटी
  [Vakyansh] Majestic ... ✓  → मैजिस टरिक से बीएमटी सेले कर जाता
  [Vakyansh] Silk Board ... ✓  → सिलरोड परट्रैफिक बहुत
  [Vakyansh] Bellandur ... ✓  → बैलडूल रोड पर फ्लैट लिए
  [Vakyansh] Sarjapur ... ✓  → सरजपुर रोड पर काम करता हूं
  [Vakyansh] Bommanahalli ... ✓  → 
  [Vakyansh] KR Puram ... ✓  → केरपुर  रेलवे स्टेशन के पास  जल्दी आजा
  [Vakyansh] Peenya ... ✓  → इंस्ट्रिय एरिया में फैक्टरी
  [Vakyansh] Yeshwanthpur ... ✓  → यट्रेन  नहीं  मेरेको जल्द
  [Vakyansh] Indiranagar_2 ... ✓  → मेरा घर इंद्रा नगर के पास ही है
  [Vakyansh] Bellandur_2 ... ✓  → बैलडू लेके पास फ्लैट लीहै
  [Vakyansh] Sarjapur_2 ... ✓  → सरजापुर   खड़जल
  [Vakyansh] Yeshwanthpur_2 ... ✓  → 

Saved → vakyansh_transcriptions.json


### Combining and Saving Model Results

Now that both Whisper and Vakyansh models have processed the audio files, the next step is to evaluate Vakyansh's output and then combine the results from both models into a unified dataset for further analysis. This section performs the evaluation, merges the data, and saves the raw and summarized results, along with combined transcriptions, to CSV and JSON files.

In [ ]:
# Ensure vakyansh_transcriptions is available (it's named 'results' from the previous Vakyansh cell output)
vakyansh_transcriptions = results

# ── Save combined results (Whisper + Vakyansh) ──────────────────────────────
import csv, json, os
from pathlib import Path

MODEL_NAME_VAKYANSH = "vakyansh"   # change if needed

# 1. Evaluate Vakyansh the same way Whisper was evaluated
vakyansh_rows    = evaluate(MODEL_NAME_VAKYANSH, vakyansh_transcriptions)
vakyansh_summary = aggregate_metrics(vakyansh_rows)

# 2. Load existing whisper rows (already computed earlier in this notebook)
#    'rows' contains the Whisper results from cell Y5cPi0RnIrmP
all_rows = rows + vakyansh_rows

# 3. Write results_raw.csv  (one row per sample, all models)
out_dir = Path(RESULTS_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "model", "locality", "canonical", "condition", "speaker",
    "ground_truth", "hypothesis",
    "wer", "cer", "entity_exact", "entity_fuzzy", "entity_ratio",
    "latency_s", "error",
]

raw_path = out_dir / "results_raw.csv"
with open(raw_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    w.writeheader()
    w.writerows(all_rows)

# 4. Write summary.csv  (one row per model)
whisper_summary = aggregate_metrics(rows)   # Whisper aggregate

summary_rows = [
    {"model": "whisper-large-v3",  **whisper_summary},
    {"model": MODEL_NAME_VAKYANSH, **vakyansh_summary},
]

summary_path = out_dir / "summary.csv"
with open(summary_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["model", "avg_wer", "avg_cer",
                                       "entity_exact_pct", "entity_fuzzy_pct",
                                       "num_samples"])
    w.writeheader()
    w.writerows(summary_rows)

# 5. Write combined transcriptions JSON
combined_json = {
    "whisper-large-v3": {k: v["transcript"] for k, v in transcriptions.items()},
    MODEL_NAME_VAKYANSH: {k: v["transcript"] for k, v in vakyansh_transcriptions.items()},
}
json_path = out_dir / "transcriptions_raw.json"
json_path.write_text(json.dumps(combined_json, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"✓ results_raw.csv   → {raw_path}  ({len(all_rows)} rows)")
print(f"✓ summary.csv       → {summary_path}")
print(f"✓ transcriptions_raw.json → {json_path}")
print_summary_table({"whisper-large-v3": whisper_summary, MODEL_NAME_VAKYANSH: vakyansh_summary})


✓ results_raw.csv   → /content/results/results_raw.csv  (48 rows)
✓ summary.csv       → /content/results/summary.csv
✓ transcriptions_raw.json → /content/results/transcriptions_raw.json

Model                      Avg WER  Avg CER  EMR Exact  EMR Fuzzy
----------------------------------------------------------------------
whisper-large-v3             1.083    0.860       4.2%       4.2%
vakyansh                     1.013    0.927       0.0%       0.0%
